# Sample 09: v0.4.0 新機能ガイド (v0.4.0 New Features Guide)

Binary Master v0.4.0 で追加された最新の高速化・解析支援・言語親和性向上機能を実践します。

### 主な新機能
1. **Pattern Language / Rust互換コンパクト型** (`u8`, `u16`, `u32`, `u64`, `i8`, `i16`, `s16`, `f32`, `f64` 等)
2. **ImHex Pattern Language (`.hexpat`) エクスポーター**
3. **ZeroCopyView (ゼロコピー・遅延解析ビュー)** & インプレース書き換え
4. **連続パケットストリーミング** (`iter_packets`, `iter_views`, `async_iter_packets`)
5. **ダミー・モックデータ自動生成** (`dummy()`, `generate_dummy()`)
6. **インタラクティブ TUI インスペクター** (`binary-master inspect -i`)

## 1. コンパクト型 (Pattern Language / Rust 互換)

ImHex Pattern Language や Rust の型命名規則に合わせた直感的な型アノテーションが使用できます。
従来の `UInt16` や `Int32` と完全な互換性を保ちながら、より簡潔に構造体を記述できます。

In [1]:
from binary_master import (
    FixedString,
    Magic,
    binary_struct,
    f32,
    s8,
    u8,
    u16,
    u32,
    u64,
)


@binary_struct(endian="little")
class SensorPacket:
    magic: Magic[b"SN"]
    version: u8
    node_id: u16
    sequence: u32
    timestamp: u64
    temp_offset: s8
    temperature: f32
    device_label: FixedString[8]

# インスタンス生成とシリアライズ
pkt = SensorPacket(
    version=1,
    node_id=42,
    sequence=1001,
    timestamp=1700000000,
    temp_offset=-2,
    temperature=24.5,
    device_label="NODE-A",
)
raw = pkt.to_bytes()
print(f"SensorPacket サイズ: {len(raw)} バイト")
print(pkt.hexdump())

# デシリアライズ検証
decoded = SensorPacket.from_bytes(raw)
print(f"復号成功: Node={decoded.node_id}, Temp={decoded.temperature:.1f}℃, Label={decoded.device_label}")

## 2. ImHex Pattern Language (.hexpat) エクスポーター

高機能バイナリエディタ **ImHex** でバイナリを可視化・リバースエンジニアリングするための `.hexpat` パターンスクリプトを1行で生成できます。

In [2]:
# ImHex パターンスクリプトを出力
hexpat_script = SensorPacket.to_hexpat()
print("--- Generated ImHex Pattern Script (.hexpat) ---")
print(hexpat_script)

# ファイルへの直接保存も可能
# SensorPacket.write_hexpat("sensor.hexpat")

## 3. ZeroCopyView (ゼロコピー・遅延パースビュー)

大容量データや高頻度パケットの処理時に、クラスインスタンスを割り当てずにバイナリバッファ上の特定フィールドだけを即座に参照・編集できます。
`bytearray` を渡すことでインプレース（メモリ直接）更新も行えます。

In [3]:
from binary_master import ZeroCopyView

# 1. ゼロコピー参照
view = SensorPacket.view(raw)
print(f"View による直接参照: node_id={view.node_id}, temp={view.temperature:.1f}")
print(f"View の辞書変換: {view.to_dict()}")

# 2. bytearray 上でのインプレース更新
buf = bytearray(raw)
mutable_view = ZeroCopyView(SensorPacket, buf)
mutable_view.node_id = 9999
mutable_view.temperature = 36.8

# バッファ側で値が書き換わっていることを確認
updated_pkt = SensorPacket.from_bytes(buf)
print(f"更新後パケット: Node={updated_pkt.node_id}, Temp={updated_pkt.temperature:.1f}℃")

## 4. 連続パケットストリーミング (`iter_packets` / `iter_views`)

シリアルポート通信、PCAPキャプチャ、テレメトリログなど、同じパケット構造が連続して並ぶストリームをイテレータとして効率的に走査します。

In [4]:

# 3つのパケットが連続したストリームデータを作成
packets = [
    SensorPacket(version=1, node_id=i, sequence=100 + i, timestamp=1700000000 + i, temp_offset=0, temperature=20.0 + i, device_label=f"NODE-{i}")
    for i in range(3)
]
stream_bytes = b"".join(p.to_bytes() for p in packets)

print(f"ストリーム全体サイズ: {len(stream_bytes)} バイト")

# 1. 通常のパケットストリーミング
print("\n--- iter_packets ---")
for p in SensorPacket.iter_packets(stream_bytes):
    print(f"Decoded Packet: Node={p.node_id}, Seq={p.sequence}, Temp={p.temperature:.1f}")

# 2. オブジェクト生成なしのゼロコピービュー走査
print("\n--- iter_views (Zero-Allocation) ---")
for v in SensorPacket.iter_views(stream_bytes):
    print(f"ZeroCopyView: Node={v.node_id}, Label={v.device_label}")

## 5. ダミー・モックデータ自動生成 (`dummy()`, `generate_dummy()`)

単体テスト、ファジングテスト、モックサーバー構築時に、制約（`Magic`, `Range`, `Constant`, `Enum`, `FixedString`）を満たす有効なランダムバイナリデータを即座に生成できます。

In [5]:
from binary_master import Constant, Range


@binary_struct
class DeviceTelemetry:
    magic: Magic[b"TEL"]
    protocol_version: Constant[u16, 0x0100]
    humidity_percent: Range[u8, 0, 100]
    pressure_hpa: Range[u16, 800, 1200]
    station_name: FixedString[8]

# シード指定で再現性のあるダミーデータを自動生成
dummy1 = DeviceTelemetry.dummy(seed=123)
print(f"生成ダミー1: Magic={dummy1.magic}, Version=0x{dummy1.protocol_version:04X}, 湿度={dummy1.humidity_percent}%, 気圧={dummy1.pressure_hpa}hPa, 局名={dummy1.station_name}")

# 特定のフィールドのみ上書き (オーバーライド)
dummy_custom = DeviceTelemetry.dummy(humidity_percent=55, station_name="TOKYO")
print(f"オーバーライド版: 湿度={dummy_custom.humidity_percent}%, 局名={dummy_custom.station_name}")

# そのままバイナリとしてシリアライズ・検証可能
raw_dummy = dummy_custom.to_bytes()
print(f"シリアライズ成功 ({len(raw_dummy)} bytes)")

## 6. まとめと CLI コマンド

v0.4.0 では CLI ツールも大幅に強化されました：

- **ImHex パターン出力**:
  ```bash
  binary-master export my_module:SensorPacket -l hexpat -o sensor.hexpat
  ```
- **インタラクティブ TUI インスペクター** (`-i` / `--interactive`):
  ```bash
  binary-master inspect dump.bin -s my_module:SensorPacket -i
  ```
  矢印キーでのスクロール、Tab キーでの構造体フィールド選択、フィールド位置のハイライト連動表示が利用できます。